# Reproduce the random-2-per-FP pair dataset

This notebook rebuilds the (first party, third party) pair dataset that
feeds RQ2-RQ3. It uses the same seed and the same rule as the original
pipeline, then recomputes all the summary numbers we quote for this pair
set.

The rule is simple: for each qualified first party, pick up to **two**
of its embedded third parties at random (seed = 42). First parties with
fewer than two qualifying third parties give what they have; first
parties with zero are dropped.

We also further curated the dataset to drop privacy policies that turned
out to be corrupted (cookie banners, access-denied pages, generic menus
and the like). Those conditions are re-applied below, so the notebook
reproduces both the selection step *and* the curation step.

Run top to bottom (Cell -> Run All). About one minute once the archive
is unpacked.

> *Note on field names.* The upstream pair file uses the JSON keys
> `site_etld1` and `vendor_etld1`. We leave those key names alone so
> the output matches exactly. In the prose and printed text below we
> always say *first-party* and *third-party* for the same thing.


## 1. Setup


### 1.1 Decompress the bundled dataset
Same tarball as the other notebook. Safe to re-run.


In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (DATA_DIR / 'results.jsonl').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}.'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')


### 1.2 Imports and helpers


In [ ]:
import json, re, glob, random, collections
import numpy as np
import matplotlib.pyplot as plt

MIN_WORDS    = 500
K_PER_FP     = 2
SEED         = 42
WORD_RE      = re.compile(r'\S+')

def wc_en(entry):
    '''Matches the reference pair builder exactly: always recompute from
    the cached text, never read a pre-stored word_count. Entries can have
    a cached word_count that disagrees with a fresh regex count by a word
    or two, which is enough to flip the >=500 threshold.'''
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    return len(WORD_RE.findall(entry.get('text') or ''))

def pct(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)


### 1.3 Load the underlying dataset
(Same inputs as `reproduce_findings.ipynb`.)


In [ ]:
rows = [json.loads(ln) for ln in open(DATA_DIR / 'results.jsonl')]
print(f'rows in results.jsonl               : {len(rows):>7,}')

# Match the original builder's glob iteration order exactly. dict.update()
# overwrites earlier keys, so when a URL appears in multiple shards the
# last one wins. Sorting flips which shard lands last and can change the
# qualifying set by a few entries.
tp_cache = {}
for f in glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json')):
    tp_cache.update(json.load(open(f)))
print(f'entries in third-party policy cache : {len(tp_cache):>7,}')

redisc = {e['rediscovered_from_etld1'].lower(): u
          for u, e in tp_cache.items()
          if isinstance(e, dict) and e.get('rediscovered_from_etld1')}


### Curate out corrupted privacy policies

Not every page we pulled down is a real privacy policy. Some turned out to
be cookie banners, "access denied" boilerplate, menu dumps, or login walls.
We flag a policy as **corrupted** if either:

- the text mentions **"privacy" fewer than 3 times**, or
- the text covers **fewer than 4** of these privacy topics: *privacy,
  personal information, personal data, cookie, consent, data protection,
  gdpr, ccpa, third part, retention, collect, process, your rights,
  opt-out, opt out.*

The cell below applies both conditions to the third-party policy cache
directly (we have the raw text for those) and loads a pre-computed list
of flagged first parties (their raw text lives in HPC artifact files that
are too big to ship).


In [ ]:
PRIVACY_KEYWORDS = [
    'privacy', 'personal information', 'personal data', 'cookie',
    'consent', 'data protection', 'gdpr', 'ccpa', 'third part',
    'retention', 'collect', 'process', 'your rights', 'opt-out', 'opt out',
]

def is_corrupted(text: str) -> bool:
    if not text:
        return True
    t = text.lower()
    if t.count('privacy') < 3:
        return True
    if sum(1 for kw in PRIVACY_KEYWORDS if kw in t) < 4:
        return True
    return False

# Apply the check live to third-party policies (cached text is bundled).
corrupted_tp_urls = {u for u, e in tp_cache.items()
                     if isinstance(e, dict) and is_corrupted(e.get('text', ''))}

# Same check ran on HPC over the first-party artifact texts; the result
# is bundled here.
curation = json.load(open(DATA_DIR / 'policy_curation.json'))
corrupted_fp_etld1s = set(curation.get('fp_blacklist_etld1', []))

print(f'Corrupted third-party policies flagged : {len(corrupted_tp_urls):,}')
print(f'Corrupted first parties flagged        : {len(corrupted_fp_etld1s):,}')


# qual_urls = TP policy URLs that are English, >=500 words, NOT corrupted.
qual_urls = {u for u, e in tp_cache.items()
             if (e.get('language','en')=='en'
                 and len(WORD_RE.findall(e.get('text') or '')) >= MIN_WORDS
                 and u not in corrupted_tp_urls)}
print(f'qualifying TP policy URLs           : {len(qual_urls):>7,}')


## 2. Which first parties can take part?

A first party is usable only if it meets both of these:

- **Qualifies** — English policy, at least 500 words, not on our
  first-party curation list.
- **Has inline policy text** in `results.jsonl`. The original pipeline
  also reads `artifacts/<etld1>/policy.txt` when the inline text is
  missing, but those files are too big to ship here — in practice every
  qualifying first party has inline text, so the two paths agree.


In [ ]:
qualifying_fp = {}   # fp_etld1 -> (text, wc)
for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    w = r.get('first_party_policy_word_count') or 0
    if w < MIN_WORDS:
        continue
    et = (r.get('site_etld1') or '').lower()
    if not et or et in corrupted_fp_etld1s:
        continue
    text = r.get('first_party_policy') or ''
    if not text:
        continue
    qualifying_fp[et] = (text, w)

print(f'Qualifying first parties with inline text: {len(qualifying_fp):,}')


## 3. Building the pairs: a funnel

For each qualifying first party we:

1. Collect its qualifying third parties (unique by eTLD+1, excluding its
   own domain).
2. Randomly pick up to two of them (`K_PER_FP = 2`).

The cells below show the funnel: how many third parties each first party
has to choose from -> how many first parties end up producing 0 / 1 / 2
pairs -> the total pair count.


### 3.1 Distribution of qualifying TPs per first party


In [ ]:
per_fp_tp_lists = {}

for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    fp_et = (r.get('site_etld1') or '').lower()
    if fp_et not in qualifying_fp:
        continue
    tps = []
    seen = set()
    for tp in r.get('third_parties') or []:
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet or tet in seen or tet == fp_et:
            continue
        pu = tp.get('policy_url') or redisc.get(tet)
        if not pu or pu in corrupted_tp_urls or pu not in qual_urls:
            continue
        entry = tp_cache.get(pu)
        if not isinstance(entry, dict):
            continue
        tp_wc = len(WORD_RE.findall(entry.get('text') or ''))
        if tp_wc < MIN_WORDS:
            continue
        seen.add(tet)
        tps.append((tet, pu, tp_wc, tp))
    per_fp_tp_lists[fp_et] = tps

counts = collections.Counter(len(v) for v in per_fp_tp_lists.values())
buckets = [0, 1, 2, 3, 5, 10, 20, 10**9]
print('Qualifying-TP-per-FP distribution (over the {:,} qualifying FPs)\n'.format(len(qualifying_fp)))
for lo, hi in zip(buckets[:-1], buckets[1:]):
    n = sum(c for k, c in counts.items() if lo <= k < hi)
    tag = f'{lo}' if hi == lo + 1 else (f'{lo}-{hi-1}' if hi < 10**9 else f'>={lo}')
    print(f'  {tag:<10s}  {n:>4}')

fps_zero = sum(1 for v in per_fp_tp_lists.values() if len(v) == 0)
fps_one  = sum(1 for v in per_fp_tp_lists.values() if len(v) == 1)
fps_gt1  = sum(1 for v in per_fp_tp_lists.values() if len(v) >= 2)
print()
print(f'FPs with 0 qualifying TPs (dropped): {fps_zero:>4}')
print(f'FPs with exactly 1 qualifying TP   : {fps_one:>4}')
print(f'FPs with >=2 qualifying TPs        : {fps_gt1:>4}')
print(f'FPs that will emit at least 1 pair : {fps_one + fps_gt1:>4}')
print(f'pairs expected (1*{fps_one} + 2*{fps_gt1}): '
      f'{fps_one + 2*fps_gt1:>5}')


### 3.2 Repeatable sampling (seed = 42)

Two things keep this repeatable:

- We call `random.seed(42)` once before the loop starts.
- First parties are visited in `results.jsonl` order, which is the same
  order the original pipeline used.

For each first party we call `random.sample(tps, k)` with
`k = min(2, len(tps))`. First parties with zero qualifying third parties
are skipped before the `random.sample` call — this matches the original
pipeline exactly and is important for getting the same stream of random
picks.


In [ ]:
random.seed(SEED)
reproduced_pairs = []

for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    fp_et = (r.get('site_etld1') or '').lower()
    if fp_et not in qualifying_fp:
        continue
    tps = per_fp_tp_lists.get(fp_et, [])
    if not tps:
        continue
    k = min(K_PER_FP, len(tps))
    picks = random.sample(tps, k)
    for tet, pu, tp_wc, tp in picks:
        reproduced_pairs.append({
            'site_etld1': fp_et,
            'vendor_etld1': tet,
            'vendor_entity': tp.get('entity') or '',
            'vendor_policy_url': pu,
            'site_policy_word_count': qualifying_fp[fp_et][1],
            'vendor_policy_word_count': tp_wc,
        })

fps_in_pairs = len({p['site_etld1'] for p in reproduced_pairs})
print(f'First parties emitted    : {fps_in_pairs:>5,}')
print(f'Pairs emitted            : {len(reproduced_pairs):>5,}')


## 5. Summary numbers across the pairs

Reproduces the counters in `random2_summary.json`: the most-picked
third-party eTLD+1s and entities, plus the share of pairs the top-10 of
each cover.


### 5.1 Top-10 third-party eTLD+1s


In [ ]:
tp_counts = collections.Counter(p['vendor_etld1'] for p in reproduced_pairs)
print('Top 10 third-party eTLD+1s, by pair count')
print(f"{'rank':<5}{'count':<10}{'third-party eTLD+1':<30}")
for i, (tp, c) in enumerate(tp_counts.most_common(10), 1):
    print(f'{i:<5}{c:<10,}{tp:<30}')

print(f'\nDistinct third-party eTLD+1s: {len(tp_counts):,}')


### 5.2 Top-10 third-party entities


In [ ]:
entity_counts = collections.Counter(p['vendor_entity'] for p in reproduced_pairs if p['vendor_entity'])
print('Top 10 third-party entities, by pair count')
print(f"{'rank':<5}{'count':<10}{'entity':<30}")
for i, (e, c) in enumerate(entity_counts.most_common(10), 1):
    print(f'{i:<5}{c:<10,}{e:<30}')

print(f'\nDistinct third-party entities: {len(entity_counts):,}')


### 5.3 Share covered by the top 10


In [ ]:
top10_etld1 = sum(c for _, c in tp_counts.most_common(10))
top10_ent   = sum(c for _, c in entity_counts.most_common(10))
n = len(reproduced_pairs)
print(f'Top-10 TP eTLD+1 share : {100*top10_etld1/n:5.1f}%   ({top10_etld1:,} / {n:,})')
print(f'Top-10 TP entity share : {100*top10_ent/n:5.1f}%   ({top10_ent:,} / {n:,})')


## 6. Policy-length distributions on the pairs

First-party policy length across the first parties that appear in the
pairs, and third-party policy length across the pairs themselves. The
same third party can show up in many pairs, so the third-party numbers
are weighted by how often each third party was picked.


In [ ]:
fp_wcs_in_pairs = {p['site_etld1']: p['site_policy_word_count'] for p in reproduced_pairs}
tp_wcs_in_pairs = [p['vendor_policy_word_count'] for p in reproduced_pairs]

fp_wcs = list(fp_wcs_in_pairs.values())
print(f'First-party (distinct FPs in pairs)   n={len(fp_wcs):>5,}   '
      f'median={pct(fp_wcs,50):>6,.0f}   mean={sum(fp_wcs)/len(fp_wcs):>6,.0f}   '
      f'IQR={pct(fp_wcs,25):>5,.0f}-{pct(fp_wcs,75):<5,.0f}')
print(f'Third-party (one row per pair)        n={len(tp_wcs_in_pairs):>5,}   '
      f'median={pct(tp_wcs_in_pairs,50):>6,.0f}   mean={sum(tp_wcs_in_pairs)/len(tp_wcs_in_pairs):>6,.0f}   '
      f'IQR={pct(tp_wcs_in_pairs,25):>5,.0f}-{pct(tp_wcs_in_pairs,75):<5,.0f}')


## 7. Figure - top-10 third parties

Horizontal bar chart of the ten most-picked third-party eTLD+1s. Same
compact publication style as the other notebook.


In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 0.8,
    'axes.edgecolor': '#cccccc',
    'pdf.fonttype': 42,
})
PRIMARY = '#2171b5'

top = tp_counts.most_common(10)[::-1]
labels = [t for t, _ in top]
counts = [c for _, c in top]

fig, ax = plt.subplots(figsize=(3.35, 2.8))
y = np.arange(len(labels))
ax.barh(y, counts, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, height=0.68)
for yi, c in zip(y, counts):
    ax.text(c + max(counts)*0.01, yi, f'{c:,}', va='center', ha='left', fontsize=6.5)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('Number of pairs', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7); ax.tick_params(axis='y', pad=2)
ax.grid(True, axis='x', alpha=0.25, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_xlim(0, max(counts) * 1.2)
plt.tight_layout(pad=0.3); plt.show()


## 8. Extraction manifest

The pair set above tells the extractor *what to read*. Before running
the reader on every policy we strip out duplicates:

- Two pairs that point at the same third-party URL only need that URL
  read once.
- Two different URLs that happen to serve the same text (same SHA) can
  be merged further.

This section rebuilds the row-level list from our pair set and compares
it to the original manifest shipped in `data/manifest.csv`.

> *Why we read the SHAs from the reference file.* First-party policy
> text lives in `artifacts/<etld1>/policy.txt` on HPC — too big to ship.
> We can rebuild the *row-level* list (one row per unique first party,
> one per unique third-party URL) from the bundled data. The *text-level*
> SHA dedup needs the raw text, so we read the SHA-16 column straight
> out of `manifest.csv` instead.


### 8.1 Row-level list (one row per unique document)

Built straight from our reproduced pair set:

- **FP rows** — one per unique `site_etld1`.
- **TP rows** — one per unique `vendor_policy_url`. Several TP eTLD+1s
  can share the same URL (e.g. `facebook.net` and `facebook.com` both
  point at Facebook's single privacy page).


In [ ]:
unique_fps_in_pairs = {p['site_etld1']      for p in reproduced_pairs}
unique_tp_urls      = {p['vendor_policy_url'] for p in reproduced_pairs}
manifest_rows       = len(unique_fps_in_pairs) + len(unique_tp_urls)

print(f'Unique first parties in the pair set : {len(unique_fps_in_pairs):>5,}')
print(f'Unique third-party policy URLs       : {len(unique_tp_urls):>5,}')
print(f'Manifest rows (FP rows + TP rows)    : {manifest_rows:>5,}')


### 8.2 SHA dedup (read from `manifest.csv`)

Each row of `manifest.csv` carries a `sha256_16` column — the first 16
hex characters of the text's SHA-256. Counting unique SHAs per role and
across roles tells us how many LLM calls the extractor actually makes.


In [ ]:
import csv

fp_sha_rows, tp_sha_rows = [], []
with open(REPO_ROOT / 'data' / 'manifest.csv') as fh:
    for row in csv.DictReader(fh):
        (fp_sha_rows if row['policy_source']=='first_party' else tp_sha_rows).append(row)

fp_shas = {r['sha256_16'] for r in fp_sha_rows}
tp_shas = {r['sha256_16'] for r in tp_sha_rows}
all_shas = fp_shas | tp_shas

print(f'Unique FP texts (SHA merged)           : {len(fp_shas):>5,}')
print(f'Unique TP texts (SHA merged)           : {len(tp_shas):>5,}')
print(f'Unique policies to extract, by role    : {len(fp_shas) + len(tp_shas):>5,}')
print(f'Unique policies merged across roles    : {len(all_shas):>5,}')

cross = fp_shas & tp_shas
print(f'\nTexts that appear as both FP and TP    : {len(cross):>5,}   (each saves one extraction)')


### 8.3 How much text the extractor sees

`manifest.csv` also records `word_count` per row. We summarise the
size of each role's documents so you can see the read load.


In [ ]:
fp_wcs = [int(r['word_count']) for r in fp_sha_rows]
tp_wcs = [int(r['word_count']) for r in tp_sha_rows]

print('Document word counts across the manifest')
print(f'  FP rows (n={len(fp_wcs):>5,})  '
      f'median={pct(fp_wcs,50):>6,.0f}  mean={sum(fp_wcs)/len(fp_wcs):>6,.0f}  '
      f'p90={pct(fp_wcs,90):>6,.0f}  max={max(fp_wcs):>6,}')
print(f'  TP rows (n={len(tp_wcs):>5,})  '
      f'median={pct(tp_wcs,50):>6,.0f}  mean={sum(tp_wcs)/len(tp_wcs):>6,.0f}  '
      f'p90={pct(tp_wcs,90):>6,.0f}  max={max(tp_wcs):>6,}')
print()
total_words = sum(fp_wcs) + sum(tp_wcs)
print(f'Total text going through the extractor: {total_words:,} words '
      f'across {len(fp_wcs)+len(tp_wcs):,} documents')
